# Per-horizon LightGBM experiment

Trains one quantile model (p10/p50/p90) per horizon in `[1, 7, 14, 28]`, plus optionally one Poisson point booster per horizon. Each sub-model is fit on rows where `horizon == h` only; the outer Phase-5 split rule (`target_date <= O - 56`) is preserved.

**Benchmarks to beat:**
- `lightgbm_quantile_p50` lifecycle: WAPE **0.696590**, Bias **-0.161**
- horizon-28 Bias (any model): around **-0.21** under both baseline and lifecycle versions

**Outputs go to:**
```
outputs/models/experiments/per_horizon/
outputs/reports/experiments/per_horizon/
outputs/figures/experiments/per_horizon/
```
The pre-experiment baseline and lifecycle artifacts are never overwritten.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import pandas as pd
pd.options.display.float_format = '{:.4f}'.format

from seercast.training.run_per_horizon_experiment import run as run_per_horizon
REPO_ROOT

## 1. Run the experiment

Trains 3 backtest origins x 4 horizons = 12 quantile sub-models (each producing p10/p50/p90), plus 12 point sub-models. Then runs the diagnostics layer on the per-horizon outputs and builds the three-way before-vs-after table.

In [ ]:
result = run_per_horizon()
overall = result['overall']
by_horizon = result['by_horizon']
paths = result['paths']
overall

## 2. Headline question: did WAPE drop below 0.6966?

In [ ]:
wape_pivot = overall.pivot_table(
    index='model', columns='version', values='WAPE'
)
wape_pivot

## 3. Bias: did horizon-28 underforecasting improve?

Quantile p50 across versions, broken out by horizon. Bias closer to 0 is better.

In [ ]:
is_p50 = by_horizon['model'].str.contains('quantile_p50', regex=False)
p50 = by_horizon[is_p50].pivot_table(
    index='horizon', columns='version', values='Bias'
)
p50

## 4. WAPE by horizon

In [ ]:
by_horizon[is_p50].pivot_table(
    index='horizon', columns='version', values='WAPE'
)

## 5. RMSE trade-off

In [ ]:
overall.pivot_table(index='model', columns='version', values='RMSE')

## 6. p10-p90 coverage (did the conservative intervals tighten?)

In [ ]:
import pandas as pd
unc = pd.read_csv(paths['uncertainty_diagnostics'])
unc[unc['dimension'] == 'horizon'][[
    'value','n','coverage_p10_p90','interval_width_p10_p90','relative_interval_width_p10_p90','crossing_rate'
]]

**Reminder.** Whatever the numbers show is the answer. Per-horizon may help, may hurt, may be flat. We report what the matched-grid comparison says.